In [1]:
import os
import urllib.request

In [2]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

In [28]:
with open("the-verdict.txt", "r", encoding="UTF-8") as f:
  raw_text = f.read()

In [29]:
raw_text

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [5]:
len(raw_data)

20479

In [7]:
import re

In [12]:
text = "Hello world, what happen"
result = re.split(r"(\s)", text)
print(result)

['Hello', ' ', 'world,', ' ', 'what', ' ', 'happen']


In [13]:
## Seperate punctuation

result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ' ', 'world', ',', '', ' ', 'what', ' ', 'happen']


In [17]:
print(text.split())

['Hello', 'world,', 'what', 'happen']


In [30]:
text = "Hello world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item for item in result if item.strip()]
print(result)

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--', 'deploring', 'his', 'unaccountable', 'abdication', '.', '"', 'Of', 'course', 'it', "'", 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'", 'way', 'up', ';', 'but', 'I', 'don', "'", 't', 'think', 'of', 'that', ',

In [31]:
len(result)

4690

In [32]:
result[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

In [35]:
## Unique Word

all_words = sorted(set(result))
vocab_size = len(all_words)
print(vocab_size)

1130


In [37]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [40]:
class SimpleTokenizerV1:

  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode_text(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
            item.strip() for item in preprocessed if item.strip()
      ]

    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # Replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text


In [45]:
tokeinzer = SimpleTokenizerV1(vocab)

In [48]:
text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""

ids = tokeinzer.encode_text(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [51]:
string = tokeinzer.decode(ids)
print(string)

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [52]:
text = "Hello, I would prefer coffee over tea"
tokeinzer.encode_text(text)

KeyError: 'Hello'

In [54]:
all_words = sorted(list(set(result)))
all_words.extend(["<|endoftext|>", "<|UNK|>"])

In [55]:
len(all_words)

1132

In [58]:
print(all_words[-5:])

['younger', 'your', 'yourself', '<|endoftext|>', '<|UNK|>']


In [59]:
tokenized_verson = {token:ids for ids, token in enumerate(all_words)}

In [60]:
class SimpleTokenizerV2:

  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode_text(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

    preprocessed = [
            item.strip() for item in preprocessed if item.strip()
      ]

    preprocessed = [
        item if item in self.str_to_int else "<|UNK|>" for item in preprocessed
    ]

    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    # Replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [63]:
tokenv2 = SimpleTokenizerV2(vocab)

text = "Hello, I would prefer coffee over tea"
ids = tokenv2.encode_text(text)
ids

KeyError: '<|UNK|>'